# 01. Objektif Bisnis dan Batasan Sistem

Perburukan kondisi pada gangguan jiwa berat jarang datang mendadak dalam semalam. Jauh sebelum keluhan memuncak, kebiasaan sehari-hari biasanya bergeser pelan selama berhari-hari hingga berminggu-minggu. Karena kunjungan ke layanan kesehatan umumnya berjarak dua minggu sampai satu bulan, rentang waktu di antara dua pertemuan inilah yang kerap luput dari pengamatan.

Dari konteks tersebut, pemantauan keseharian berhadapan dengan **lima persoalan nyata di lapangan**:
1. **Patokan pribadi vs patokan umum yang membingungkan:** perubahan pola tidur atau gerak sering baru terlihat bila dibandingkan dengan kebiasaan individu itu sendiri (bukan angka rata-rata orang lain), namun pengguna juga sering bertanya apakah ritme istirahatnya masih wajar dibanding kelompok sebaya seusianya.
2. **Perbedaan antara perasaan subyektif dan aktivitas nyata:** catatan perasaan harian (*self-report*) tidak selalu sejalan dengan aktivitas fisik yang benar-benar dijalani hari itu.
3. **Sinyal berubah tidak serentak dan arah ke depan tidak jelas:** kadang pola tidur bergeser duluan sementara komunikasi masih ramai, dan pengguna sulit membayangkan bagaimana kecenderungan dua hingga empat minggu ke depan jika pola itu dibiarkan.
4. **Ketakutan terhadap aplikasi yang menghakimi:** tampilan medis yang kaku, diagnosis sepihak, atau kalimat penyemangat palsu (*toxic positivity*) justru membuat pengguna menjauh dan enggan mencatat.
5. **Kebutuhan tiap pihak tidak sama:** pengguna butuh refleksi tenang, keluarga atau pendamping rentan kelelahan bila terus menerima alarm palsu, sedangkan klinisi membutuhkan kepekaan skrining awal sebelum sesi konsultasi.

Berdasarkan lima persoalan itu, **Jeda** ditetapkan sebagai sistem pemantauan dini berbasis sinyal perilaku pasif yang ringan untuk komputer CPU 8 GB RAM, dengan dua batas mutlak: **tidak memberikan diagnosis klinis** dan **tidak memakai AI generatif untuk mengarang pesan ke pengguna**. Mari muat konfigurasi dasar proyek dari `config.json` terlebih dahulu.

In [1]:
import json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

with open(ROOT / "config.json", "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

SEED = int(CONFIG["project"]["seed"])
REPORTS_DIR = ROOT / CONFIG["paths"]["reports"]
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Proyek : {CONFIG['project']['name']}: {CONFIG['project']['subtitle']}")
print(f"Target : {CONFIG['project']['compute_mode'].upper()} | {CONFIG['project']['ram_budget_gb']} GB RAM | SEED={SEED}")
print(f"Peran Kebijakan Peringatan: {CONFIG['alert_roles']}")

Proyek : Jeda: Early Behavioral Shift Detection for Severe Mental Illness
Target : CPU | 8 GB RAM | SEED=42
Peran Kebijakan Peringatan: ['Individu', 'Pendamping', 'Klinisi']


**Pembacaan Output Konfigurasi:**
Konfigurasi aktif pada target **CPU dengan batas RAM 8 GB** dan pengacak tetap `SEED=42`. Tiga peran kebijakan peringatan juga resmi terdaftar dari konfigurasi: `Individu`, `Pendamping`, dan `Klinisi`.

## Dari Lima Persoalan Menuju Sembilan Fitur Produk dan Kebijakan Tiga Peran

Agar kelima persoalan di atas terjawab tuntas secara terstruktur, sistem dirancang ke dalam **sembilan fitur produk** berikut:

| Persoalan di Lapangan | Nama Fitur di Jeda | Fungsi Sederhana bagi Pengguna |
|---|---|---|
| Sulit mengenali apakah rutinitas harian masih seperti biasa | **Cermin** | Membandingkan pola harian terbaru dengan kebiasaan pribadi minggu-minggu sebelumnya, lengkap dengan konteks akhir pekan atau masa padat. |
| Rasa ingin tahu apakah jam istirahat masih dalam rentang wajar sebaya | **Kurva Sehat** | Menempatkan durasi istirahat di pita referensi populasi berdasarkan usia dan jenis kelamin (bukan label sehat/sakit). |
| Catatan perasaan harian tidak selalu selaras dengan aktivitas fisik | **Jurang Insight** | Menunjukkan sisi-ke-sisi antara catatan mandiri dan pola aktivitas terukur secara netral tanpa menuduh. |
| Butuh rangkuman status dini yang tenang tanpa angka teknis rumit | **Deteksi Pergeseran** | Mengubah pola sinyal perilaku menjadi empat status sederhana: `Stabil`, `Mulai Bergeser`, `Perlu Perhatian`, atau `Jeda`. |
| Ingin melihat arah kecenderungan jangka pendek bila ritme tidak berubah | **Waktu** | Memberi gambaran arah peluang perubahan dua hingga empat minggu ke depan. |
| Kebiasaan mana yang tampak bergeser lebih awal | **Pemicu** | Mengurutkan tiga kanal perilaku yang paling awal bergerak berbeda sebagai catatan observasional. |
| Kebiasaan sehari-hari bisa bergerak saling bertolak belakang | **Inkonsistensi Antar-Sinyal** | Menyorot bila ada kanal perilaku (misalnya tidur vs aktivitas) yang bergerak tidak sejalan. |
| Butuh dorongan kalimat yang aman sesuai kondisi | **Tombol Semangat** | Menyajikan pesan dukungan dari pustaka terkurasi aturan tetap, yang otomatis dialihkan ke ajakan mencari dukungan saat risiko tinggi atau data kurang. |
| Pihak berbeda butuh tingkat kewaspadaan berbeda + anti-alarm palsu | **Kebijakan Peringatan** | Membedakan ambang respons untuk **Individu**, **Pendamping**, dan **Klinisi**, serta menahan penilaian (**ABSTAIN / Jeda**) jika data kurang. |

Membagi aturan respons menjadi tiga peran dan satu gerbang penahan menjadi kunci menyeimbangkan risiko **terlambat mendeteksi (*missed shift*)** melawan **kelelahan alarm palsu (*alarm fatigue*)**. Sel kode di bawah menuliskan tabel kebutuhan tiap peran beserta matriks biaya risiko kesalahannya.

In [2]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

stakeholders_df = pd.DataFrame([
    {
        "peran_pengguna": "Individu",
        "kebutuhan_utama": "Refleksi dini yang tenang atas pergeseran pola harian tanpa stigma atau angka teknis membingungkan.",
        "kedalaman_informasi": "Status 4 tingkat (Stabil / Mulai Bergeser / Perlu Perhatian / Jeda ABSTAIN), Cermin mingguan, Pita Kurva Sehat, Jurang Insight, 3 sinyal awal Pemicu, Inkonsistensi Antar-Sinyal, dan Tombol Semangat aman.",
        "batas_kebijakan": "Ambang paling ramah refleksi mandiri; tanpa diagnosis medis atau bahasa menyalahkan.",
    },
    {
        "peran_pengguna": "Pendamping",
        "kebutuhan_utama": "Mengetahui kapan dukungan hangat atau pengecekan kabar dibutuhkan tanpa melanggar privasi detail pengguna.",
        "kedalaman_informasi": "Ringkasan status kesiapan & arah horizon waktu 2-4 minggu setelah persetujuan awal pengguna; tanpa rincian lokasi atau komunikasi sensitif.",
        "batas_kebijakan": "Ambang lebih konservatif (alarm palsu diminimalkan) untuk mencegah kelelahan peringatan keluarga atau pendamping.",
    },
    {
        "peran_pengguna": "Klinisi",
        "kebutuhan_utama": "Konteks awal pola perilaku sebelum sesi konsultasi untuk mendukung evaluasi klinis profesional.",
        "kedalaman_informasi": "Ringkasan lintasan mingguan seluruh sinyal pasif, konteks diskordansi Jurang Insight, kecenderungan Waktu 2-4 minggu, dan status kecukupan data (ABSTAIN).",
        "batas_kebijakan": "Ambang sensitivitas lebih terukur untuk skrining dini; tetap bersifat alat bantu observasi, bukan vonis diagnostik.",
    },
])

cost_matrix_df = pd.DataFrame([
    {
        "kejadian": "Missed Shift (Pergeseran terlewat / tidak terdeteksi)",
        "dampak_individu": "Kesempatan jeda atau penyesuaian dukungan lebih awal terlewat hingga gejala memberat.",
        "dampak_pendamping_klinik": "Kehilangan jendela observasi dini sebelum episode akut.",
        "mitigasi_desain": "Kalibrasi probabilitas out-of-sample, evaluasi berbasis episode (Intent-to-Screen Recall & Lead Time), serta pemantauan urutan sinyal awal Pemicu.",
    },
    {
        "kejadian": "Alarm Fatigue (Peringatan palsu berulang)",
        "dampak_individu": "Memicu kecemasan, rasa diawasi, atau mematikan pemantauan.",
        "dampak_pendamping_klinik": "Pendamping atau klinisi mengabaikan peringatan karena terlalu sering salah.",
        "mitigasi_desain": "Gerbang ABSTAIN saat data kurang atau ketidakpastian tinggi, penggabungan alarm beruntun di evaluasi episode, dan pembedaan ambang 3 peran Kebijakan Peringatan.",
    },
    {
        "kejadian": "Jeda / ABSTAIN (Data kurang atau model ragu)",
        "dampak_individu": "Tidak dipaksa menerima kesimpulan keliru; tetap mendapat pesan netral yang hangat.",
        "dampak_pendamping_klinik": "Transparansi batas keyakinan sistem dan kejujuran epistemik.",
        "mitigasi_desain": "Syarat hari valid minimum (dibekukan pada tahap seleksi data) + ambang ketidakpastian ensemble dan kalibrasi.",
    },
])

display(stakeholders_df)
display(cost_matrix_df)

,peran_pengguna,kebutuhan_utama,kedalaman_informasi,batas_kebijakan
0,Individu,Refleksi dini yang tenang atas pergeseran pola harian tanpa stigma atau angka teknis membingungkan.,"Status 4 tingkat (Stabil / Mulai Bergeser / Perlu Perhatian / Jeda ABSTAIN), Cermin mingguan, Pita Kurva Sehat, Jurang Insight, 3 sinyal awal Pemicu, Inkonsistensi Antar-Sinyal, dan Tombol Semangat aman.",Ambang paling ramah refleksi mandiri; tanpa diagnosis medis atau bahasa menyalahkan.
1,Pendamping,Mengetahui kapan dukungan hangat atau pengecekan kabar dibutuhkan tanpa melanggar privasi detail pengguna.,Ringkasan status kesiapan & arah horizon waktu 2-4 minggu setelah persetujuan awal pengguna; tanpa rincian lokasi atau komunikasi sensitif.,Ambang lebih konservatif (alarm palsu diminimalkan) untuk mencegah kelelahan peringatan keluarga atau pendamping.
2,Klinisi,Konteks awal pola perilaku sebelum sesi konsultasi untuk mendukung evaluasi klinis profesional.,"Ringkasan lintasan mingguan seluruh sinyal pasif, konteks diskordansi Jurang Insight, kecenderungan Waktu 2-4 minggu, dan status kecukupan data (ABSTAIN).","Ambang sensitivitas lebih terukur untuk skrining dini; tetap bersifat alat bantu observasi, bukan vonis diagnostik."


,kejadian,dampak_individu,dampak_pendamping_klinik,mitigasi_desain
0,Missed Shift (Pergeseran terlewat / tidak terdeteksi),Kesempatan jeda atau penyesuaian dukungan lebih awal terlewat hingga gejala memberat.,Kehilangan jendela observasi dini sebelum episode akut.,"Kalibrasi probabilitas out-of-sample, evaluasi berbasis episode (Intent-to-Screen Recall & Lead Time), serta pemantauan urutan sinyal awal Pemicu."
1,Alarm Fatigue (Peringatan palsu berulang),"Memicu kecemasan, rasa diawasi, atau mematikan pemantauan.",Pendamping atau klinisi mengabaikan peringatan karena terlalu sering salah.,"Gerbang ABSTAIN saat data kurang atau ketidakpastian tinggi, penggabungan alarm beruntun di evaluasi episode, dan pembedaan ambang 3 peran Kebijakan Peringatan."
2,Jeda / ABSTAIN (Data kurang atau model ragu),Tidak dipaksa menerima kesimpulan keliru; tetap mendapat pesan netral yang hangat.,Transparansi batas keyakinan sistem dan kejujuran epistemik.,Syarat hari valid minimum (dibekukan pada tahap seleksi data) + ambang ketidakpastian ensemble dan kalibrasi.


**Makna Matriks Peran dan Biaya Kesalahan:**
Tabel pertama memperjelas kedalaman informasi kesembilan fitur di tiap peran: **Individu** melihat Cermin, Kurva Sehat, Jurang Insight, 3 sinyal awal Pemicu, Inkonsistensi, serta Tombol Semangat; **Pendamping** mendapat ringkasan status yang lebih selektif tanpa membocorkan detail lokasi atau komunikasi; sementara **Klinisi** memperoleh rangkuman lintasan untuk konteks sebelum konsultasi. Tabel kedua menegaskan bagaimana gerbang `ABSTAIN` dan ambang terpisah meredam *alarm fatigue* tanpa menutup mata pada risiko *missed shift*.

## Menetapkan Tangga Klaim Ilmiah dan Kriteria Keberhasilan

Sebelum kita melihat satu baris data pun, perlu ditetapkan aturan agar proyek tetap jujur terhadap hasilnya nanti. Setiap temuan pada sembilan fitur di atas kelak ditempatkan ke dalam **Tangga Klaim** tiga tingkat:
- **Tingkat A:** temuan deskriptif, aturan deterministik, atau evaluasi yang masih memakai target pengganti (*proxy*).
- **Tingkat B:** teruji keluar-lipatan (*out-of-subject*) pada orang baru yang tidak ikut dilatih.
- **Tingkat C:** tetap bertahan saat diuji ke dataset independen kedua tanpa menyetel ulang model.

Sel berikut merumuskan kriteria keberhasilan akhir proyek dan menyimpan catatan kesepakatan desain awal ke `reports/decision_log.md`.

In [3]:
success_criteria = [
    "1. Seluruh notebook alur kerja (00-11) lolos Restart & Run All di CPU tanpa data leakage.",
    "2. Seluruh 9 fitur utama (Cermin, Kurva Sehat, Jurang Insight, Deteksi Pergeseran, Waktu, Pemicu, Inkonsistensi Antar-Sinyal, Tombol Semangat, Kebijakan Peringatan) menghasilkan artefak nyata dan dicatat jujur beserta tingkat klaimnya di reports/metrics_registry.csv.",
    "3. Web App streamlit run app.py menampilkan seluruh fitur dengan estetika tenang, nol angka teknis membingungkan di mode pengguna, lolos audit kata terlarang, serta menyediakan saklar Mode Data Scientist.",
]

print("Kriteria Sukses Proyek Jeda:")
for item in success_criteria:
    print(" ", item)

decision_log_path = REPORTS_DIR / "decision_log.md"
decision_log_content = """# Decision Log: Proyek Jeda

| Tahap | Keputusan | Alasan & Dampak |
|---|---|---|
| Setup Lingkungan | Menggunakan `matplotlib==3.9.1.post1` di `requirements.txt` | `matplotlib==3.9.1` tidak menyediakan wheel biner `cp311-win_amd64` di PyPI sehingga memicu kompilasi sumber C++ pada Windows tanpa MSVC. `3.9.1.post1` adalah rilis wheel resmi pengganti `3.9.1`. |
| Objektif Bisnis | Memisahkan **Kebijakan Peringatan** menjadi 3 peran (`Individu`, `Pendamping`, `Klinisi`) + gerbang `ABSTAIN` | Menyeimbangkan risiko *Missed Shift* dan *Alarm Fatigue* secara transparan tanpa diagnosis klinis. |
| Objektif Bisnis | Mengunci batas non-diagnostik dan non-LLM generatif untuk pesan pengguna | Menjaga keamanan pesan dukungan dan kejujuran epistemik sistem. |
"""
decision_log_path.write_text(decision_log_content, encoding="utf-8")

print(f"Artefak ditulis: {decision_log_path}")

Kriteria Sukses Proyek Jeda:
  1. Seluruh notebook alur kerja (00-11) lolos Restart & Run All di CPU tanpa data leakage.
  2. Seluruh 9 fitur utama (Cermin, Kurva Sehat, Jurang Insight, Deteksi Pergeseran, Waktu, Pemicu, Inkonsistensi Antar-Sinyal, Tombol Semangat, Kebijakan Peringatan) menghasilkan artefak nyata dan dicatat jujur beserta tingkat klaimnya di reports/metrics_registry.csv.
  3. Web App streamlit run app.py menampilkan seluruh fitur dengan estetika tenang, nol angka teknis membingungkan di mode pengguna, lolos audit kata terlarang, serta menyediakan saklar Mode Data Scientist.
Artefak ditulis: ./reports\decision_log.md


**Artefak Terkunci ke `reports/decision_log.md`:**
Kriteria sukses bisnis dan produk resmi dicatat ke `reports/decision_log.md`, termasuk prinsip bahwa setiap fitur yang tidak bekerja baik saat pengujian wajib dilaporkan jujur sesuai tingkat klaimnya alih-alih dipoles agar tampak berhasil.

## Penutup Tahap 1

Objektif bisnis **Jeda** kini lengkap dan runtut: dari lima persoalan pemantauan harian lahir **sembilan fitur produk** yang disajikan ke **tiga peran** (`Individu`, `Pendamping`, `Klinisi`) di bawah pagar non-diagnostik dan gerbang aman **Jeda (`ABSTAIN`)**.

**Pertanyaan lanjutan menuju Tahap 2 (`02_technical_objectives.ipynb`):**
Kesembilan fitur yang kita cetuskan di sini baru terdefinisi dari sisi manfaat bagi pengguna. Bagaimana menerjemahkan setiap fitur dari sembilan fitur tersebut menjadi formulasi algoritmik *machine learning*, memilih tolok ukur sederhana (*baseline*) untuk dilawan, menetapkan metrik pengukurnya, dan menulis hipotesis yang berani gagal sebelum membuka data mentah?